# SSURGO Soil Data Cleaning

**Objective:** Build one soil polygon layer for the Salinas River watershed (HUC8 18060005) with the SSURGO attributes used as model features: slope, depth to bedrock, available water storage, drainage class, and hydrologic soil group.

**Inputs (local only, not tracked by Git)**
- SSURGO survey downloads (one folder per survey area), each with `spatial/soilmu_a_*.shp` and `tabular/comp.txt`, `muaggatt.txt`, `mstabcol.txt`
- NHD HU8 geodatabase `NHD_H_18060005_HU8_GDB.gdb` (layer `WBDHU8`), used as the watershed boundary

**Output (local only):** `data/soils/soils_huc8_attributes.gpkg`, one row per soil polygon with its map-unit attributes. It is read by `notebooks/merge/all_merge_final.ipynb`, which computes area-weighted features within each well buffer.

Set the two input paths in the next cell before running.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import geopandas as gpd

# Make src importable from any notebook folder (identical in every notebook)
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())))
from src import paths

PROJECT_ROOT = paths.ROOT

# ===== Inputs (raw downloads, not in git; see the README's data sources) =====
SOIL_FOLDER = paths.RAW_SSURGO      # folder that holds the CA### SSURGO survey folders
HUC8_GDB = paths.HUC8_GDB           # USGS NHD geodatabase for HU8 18060005

# ===== Output =====
OUTPUT_PATH = paths.SOILS

paths.require(SOIL_FOLDER, HUC8_GDB)
print("SSURGO folder:", SOIL_FOLDER)
print("HUC8 geodatabase:", HUC8_GDB)
print("Output:", OUTPUT_PATH)


SSURGO folder: /mnt/c/Users/Charl/Documents/WORK/ERDOS_SCHOOL_FILES/fall-2026-infrastructure-placement/data/datasets/ssurgo_soildata
HUC8 geodatabase: /mnt/c/Users/Charl/Documents/WORK/ERDOS_SCHOOL_FILES/fall-2026-infrastructure-placement/data/NHD_H_18060005_HU8_GDB/NHD_H_18060005_HU8_GDB.gdb
Output: /mnt/c/Users/Charl/Documents/WORK/ERDOS_SCHOOL_FILES/fall-2026-infrastructure-placement/data/soils/soils_huc8_attributes.gpkg


## 1. Soil polygons clipped to the watershed

Read every SSURGO survey's map-unit polygons, combine them, and keep only the area inside the Salinas HUC8 boundary.

In [2]:
# Find and combine all SSURGO soil polygon shapefiles
soil_shps = sorted(SOIL_FOLDER.rglob("soilmu_a_*.shp"))
soil_layers = [gpd.read_file(path) for path in soil_shps]

soils = gpd.GeoDataFrame(
    pd.concat(soil_layers, ignore_index=True),
    crs=soil_layers[0].crs
)
print(f"{len(soil_shps)} survey areas | {len(soils):,} polygons")

5 survey areas | 27,653 polygons


In [3]:
# Load HUC8 watershed boundary
huc8 = gpd.read_file(
    HUC8_GDB,
    layer="WBDHU8"
)

# Match coordinate systems
huc8 = huc8.to_crs(soils.crs)

# Clip soils to the watershed
soils_huc8 = gpd.clip(soils, huc8)

soils_huc8.shape

(15906, 5)

## 2. SSURGO attribute tables

The tabular files have no headers; column names come from the SSURGO metadata table `mstabcol.txt`, which is identical in every survey. Only map units inside the watershed are kept.

In [4]:
# Find all SSURGO component tables 
comp_files = sorted(SOIL_FOLDER.rglob("comp.txt"))

# Read and combine all SSURGO component tables
component_tables = [
    pd.read_csv(path, sep="|", header=None)
    for path in comp_files
]

components = pd.concat(
    component_tables,
    ignore_index=True
)

components.head()

,0,1,2,3,4,5,6,7,8,9,...,99,100,101,102,103,104,105,106,107,108
0,NaN,2,NaN,Linne,Series,No,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,455540,29305552
1,NaN,2,NaN,Diablo,Series,No,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,455540,29305553
2,NaN,2,NaN,Los Osos,Series,No,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,455540,29305554
3,NaN,2,NaN,Gazos,Series,No,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,455540,29305555
4,NaN,2,NaN,Millsholm,Series,No,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,455540,29305556


In [5]:
# Load SSURGO metadata
meta = pd.read_csv(
    next(SOIL_FOLDER.rglob("mstabcol.txt")),
    sep="|",
    header=None
)

# Get the real column names for the component table
component_cols = (
    meta[meta[0] == "component"]
    .sort_values(1)[2]
    .tolist()
)

# Assign the column names
components.columns = component_cols
components.columns

Index(['comppct_l', 'comppct_r', 'comppct_h', 'compname', 'compkind',
       'majcompflag', 'otherph', 'localphase', 'slope_l', 'slope_r',
       ...
       'flsoilleachpot', 'flsoirunoffpot', 'fltemik2use', 'fltriumph2use',
       'indraingrp', 'innitrateleachi', 'misoimgmtgrp', 'vasoimgtgrp', 'mukey',
       'cokey'],
      dtype='str', length=109)

In [6]:
# Get only the MUKEY values that are inside the HUC8
huc8_mukeys = soils_huc8["MUKEY"].astype(str).unique()

# Keep only components belonging to soil map units inside the HUC8
components_huc8 = components[
    components["mukey"].astype(str).isin(huc8_mukeys)
]

components_huc8.shape

(3074, 109)

In [7]:
# Find, read and combine the map-unit aggregate tables (one per survey)
muaggatt_files = sorted(SOIL_FOLDER.rglob("muaggatt.txt"))
muaggatt_tables = [
    pd.read_csv(path, sep="|", header=None)
    for path in muaggatt_files
]

muaggatt = pd.concat(
    muaggatt_tables,
    ignore_index=True
)

In [8]:
muaggatt_cols = (
    meta[meta[0] == "muaggatt"]
    .sort_values(1)[2]
    .tolist()
)

muaggatt.columns = muaggatt_cols

In [9]:
muaggatt_huc8 = muaggatt[
    muaggatt["mukey"].astype(str).isin(huc8_mukeys)
]

muaggatt_huc8.shape

(492, 40)

## 3. Select soil attributes

Candidate attributes related to nitrate leaching and hydrology are selected first, then checked for missing values at the map-unit level. Water-table depth and flooding frequency were dropped after the coverage check; the five attributes kept are listed below.

In [10]:
# Select soil variables relevant to nitrate transport and hydrology
soil_vars = [
    "mukey",
    "slopegradwta",
    "brockdepmin",
    "wtdepannmin",
    "aws0150wta",
    "drclassdcd",
    "hydgrpdcd",
    "flodfreqdcd"
]

soils_attributes = muaggatt_huc8[soil_vars]

soils_attributes.head()

,mukey,slopegradwta,brockdepmin,wtdepannmin,aws0150wta,drclassdcd,hydgrpdcd,flodfreqdcd
0,455537,6.0,91.0,NaN,14.56,Well drained,D,NaN
1,455538,12.0,91.0,NaN,14.56,Well drained,D,NaN
2,455539,23.0,91.0,NaN,14.56,Well drained,D,NaN
3,455540,40.0,91.0,NaN,14.56,Well drained,D,NaN
4,455541,38.0,43.0,NaN,12.00,Well drained,D,NaN


In [11]:
# Keep soil predictors with good data coverage
soil_vars_final = [
    "mukey",# mukey = unique ID for each SSURGO soil map unit
    "slopegradwta",# slopegradwta = weighted average slope gradient for the map unit
    "brockdepmin", # brockdepmin = minimum depth to bedrock within the map unit
    "aws0150wta", # aws0150wta = weighted average available water storage in the top 150 cm of soil
    "drclassdcd",# drclassdcd = dominant drainage class of the map unit
    "hydgrpdcd"# hydgrpdcd = dominant hydrologic soil group of the map unit
]
soils_attributes = muaggatt_huc8[soil_vars_final]

## 4. Join attributes to polygons and check the result

In [12]:
# Make MUKEY the same data type in both tables
soils_huc8["MUKEY"] = soils_huc8["MUKEY"].astype(str)
soils_attributes["mukey"] = soils_attributes["mukey"].astype(str)

# Join soil attributes to the HUC8 soil polygons
soils_model = soils_huc8.merge(
    soils_attributes,
    left_on="MUKEY",
    right_on="mukey",
    how="left"
)

soils_model.shape

(15906, 11)

In [13]:
# Remove duplicate join key
soils_model = soils_model.drop(columns=["mukey"])

In [14]:
# Check missing values at the unique soil map-unit level
soil_units = (
    soils_model
    .drop_duplicates("MUKEY")
)

soil_units[
    ["slopegradwta", "brockdepmin", "aws0150wta",
     "drclassdcd", "hydgrpdcd"]
].isna().sum()

slopegradwta      1
brockdepmin     234
aws0150wta       16
drclassdcd       31
hydgrpdcd        23
dtype: int64

In [15]:
# Duplicated rows and invalid geometries (both should be 0)
print("Duplicated rows:", soils_model.duplicated().sum())
print("Invalid geometries:", (~soils_model.geometry.is_valid).sum())

Duplicated rows: 0
Invalid geometries: 0


## 5. Save

In [16]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
soils_model.to_file(OUTPUT_PATH, layer="soils_huc8_attributes", driver="GPKG")
print(f"Saved {len(soils_model):,} polygons to {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")

Saved 15,906 polygons to data/soils/soils_huc8_attributes.gpkg


## 6. Conclusions

- The watershed is covered by **5 SSURGO survey areas**: **15,906 soil polygons** belonging to **492 map units**.
- Five attributes are kept: slope (`slopegradwta`), minimum depth to bedrock (`brockdepmin`), available water storage 0–150 cm (`aws0150wta`), drainage class (`drclassdcd`), and hydrologic soil group (`hydgrpdcd`).
- Missing values at the map-unit level (of 492): depth to bedrock 234, drainage class 31, hydrologic group 23, water storage 16, slope 1. Depth to bedrock is later removed in feature selection (73% missing at the well level).
- No duplicated rows or invalid geometries.
- Polygon attributes are converted to well-level, area-weighted features (500 m buffers) in `notebooks/data_merge/all_merge_final.ipynb`.